# 03 消融测试 + 安慰剂测试 (Ablation & Placebo)

## 实验概述

**目的：** 通过逐步掩码不同信息维度，找出最大的记忆触发因素；通过安慰剂测试验证模型是否真正在分析文本。

**消融测试方法：** 逐步掩码年份→实体→数字→板块，观察每步掩码对泄露指标的影响。掩码后 L 下降最多的维度 = 最大的记忆触发因素。包含 rule-based 和 LLM-based 两种掩码模式的对比。

**实体掩码方式对比：** 三种处理方式——占位符 `[实体N]`（rule-based）、LLM 自然模糊化、虚构实体替换（entity swap）。PC 高 = 模型正确忽略实体名（好），PC 低 = 模型依赖实体名（可能是记忆触发）。

**安慰剂测试方法：** 用 LLM 生成自相矛盾的虚构新闻，使理性分析者无法得出确定方向。如果模型在安慰剂数据上准确率仍高，说明它在"回忆"而非"分析"。

**关键指标：**
- PC / CI / IDS / L：见 notebook 02 定义
- 安慰剂准确率阈值：majority_class_rate + 10% = 55%（数据集 up:19, down:17, neutral:6，多数类占比 45%）

In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src import set_seed, DEFAULT_CONCURRENCY
from src.masking import extract_json_robust
from src.models import *
from src.llm_client import LLMClient
from src.news_loader import load_test_cases, load_counterfactual_variants
from src.experiment import run_counterfactual_attack, run_scoring_batch
from src.masking import apply_masking
from src.prompts import scoring_prompt, entity_swap_prompt, placebo_rewrite_prompt
from src.metrics import prediction_consistency, confidence_invariance, input_dependency_score, composite_leakage_score
from src.display_utils import show_comparison, strip_markdown_json
import json
import numpy as np
import pandas as pd
import random

set_seed()

## 1. 消融测试：逐维度掩码

In [ ]:
test_cases = load_test_cases()
variants = load_counterfactual_variants()
variant_map = {}
for v in variants:
    variant_map.setdefault(v.original_case_id, {})[v.variant_type] = v

client = LLMClient()
output_format = "5-bin"

# Rule-based ablation: progressively add masks
ablation_configs = {
    "baseline": MaskingConfig(),
    "+year": MaskingConfig(mask_year=True),
    "+year+entity": MaskingConfig(mask_year=True, mask_entity=True),
    "+year+entity+number": MaskingConfig(mask_year=True, mask_entity=True, mask_numbers=True),
    "+year+entity+number+sector": MaskingConfig(mask_year=True, mask_entity=True, mask_numbers=True, mask_sector=True),
    "full (all+role+cot+constraint)": MaskingConfig(mask_year=True, mask_entity=True, mask_numbers=True, mask_sector=True, role_play=True, cot_forced=True, extraction_constraint=True),
}

# LLM-based ablation configs
llm_configs = {
    "llm_entity": MaskingConfig(mask_entity=True, mask_mode="llm"),
    "llm_full_mask": MaskingConfig(mask_year=True, mask_entity=True, mask_numbers=True, mask_sector=True, mask_mode="llm"),
    "llm_full+role+cot+constraint": MaskingConfig(
        mask_year=True, mask_entity=True, mask_numbers=True, mask_sector=True,
        mask_mode="llm", role_play=True, cot_forced=True, extraction_constraint=True,
    ),
}

# Combine all configs
all_configs = {**ablation_configs, **llm_configs}

In [ ]:
def parse_5bin(raw: str) -> dict:
    data = extract_json_robust(raw)
    if not data:
        return {"direction": "neutral", "confidence": 0.5, "distribution": [0.2]*5}
    dist = [data.get(k, 20) for k in ["strong_bear", "weak_bear", "neutral", "weak_bull", "strong_bull"]]
    total = sum(dist)
    dist = [d/total for d in dist] if total > 0 else [0.2]*5
    bull, bear = dist[3]+dist[4], dist[0]+dist[1]
    direction = "up" if bull > bear + 0.1 else ("down" if bear > bull + 0.1 else "neutral")
    return {"direction": direction, "confidence": max(bull, bear, dist[2]), "distribution": dist}

ablation_results = []
for config_name, config in all_configs.items():
    print(f"\nRunning: {config_name}")

    orig_responses, cf_responses, task_meta = run_counterfactual_attack(
        client, config, test_cases, variant_map, output_format
    )

    for (tc, vt_name), orig_resp, cf_resp in zip(task_meta, orig_responses, cf_responses):
        orig = parse_5bin(orig_resp.raw_response)
        cf = parse_5bin(cf_resp.raw_response)
        ablation_results.append({
            "config": config_name, "case_id": tc.id, "variant_type": vt_name,
            "orig_dir": orig["direction"], "cf_dir": cf["direction"],
            "orig_conf": orig["confidence"], "cf_conf": cf["confidence"],
            "orig_dist": orig["distribution"], "cf_dist": cf["distribution"],
        })

## 2. 消融结果分析

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib
matplotlib.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False

df_abl = pd.DataFrame(ablation_results)

# 整体 metrics（所有变体类型混合）
abl_metrics = []
for config_name, group in df_abl.groupby("config"):
    pc = prediction_consistency(group["orig_dir"].tolist(), group["cf_dir"].tolist())
    consistent = [o == c for o, c in zip(group["orig_dir"], group["cf_dir"])]
    ci = confidence_invariance(group["orig_conf"].tolist(), group["cf_conf"].tolist(), consistent)
    ids = input_dependency_score(group["orig_dist"].tolist(), group["cf_dist"].tolist())
    L = composite_leakage_score(pc, ci, ids)
    abl_metrics.append({"config": config_name, "PC": pc, "CI": ci, "IDS": ids, "L": L})

abl_df = pd.DataFrame(abl_metrics)
config_order = list(all_configs.keys())
abl_df["config"] = pd.Categorical(abl_df["config"], categories=config_order, ordered=True)
abl_df = abl_df.sort_values("config")

print("Ablation results (all variant types combined):")
print(abl_df.to_string(index=False, float_format="%.3f"))

# 按 variant_type 分别展示
print("\n\nAblation results by variant type:")
for vt in VariantType:
    vt_group = df_abl[df_abl["variant_type"] == vt.value]
    if vt_group.empty:
        continue
    print(f"\n--- {vt.value} ---")
    vt_metrics = []
    for config_name, group in vt_group.groupby("config"):
        pc = prediction_consistency(group["orig_dir"].tolist(), group["cf_dir"].tolist())
        consistent = [o == c for o, c in zip(group["orig_dir"], group["cf_dir"])]
        ci = confidence_invariance(group["orig_conf"].tolist(), group["cf_conf"].tolist(), consistent)
        ids = input_dependency_score(group["orig_dist"].tolist(), group["cf_dist"].tolist())
        L = composite_leakage_score(pc, ci, ids)
        vt_metrics.append({"config": config_name, "PC": pc, "CI": ci, "IDS": ids, "L": L})
    vt_df = pd.DataFrame(vt_metrics)
    vt_df["config"] = pd.Categorical(vt_df["config"], categories=config_order, ordered=True)
    vt_df = vt_df.sort_values("config")
    print(vt_df.to_string(index=False, float_format="%.3f"))

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
x = range(len(abl_df))
width = 0.2
ax.bar([i - width for i in x], abl_df["PC"], width, label="PC", color="#F44336")
ax.bar(x, abl_df["CI"], width, label="CI", color="#FF9800")
ax.bar([i + width for i in x], abl_df["IDS"], width, label="IDS", color="#4CAF50")
ax.set_xticks(x)
ax.set_xticklabels(abl_df["config"], rotation=45, ha="right", fontsize=9)
ax.set_ylabel("Score")
ax.set_title("Ablation: Rule-based + LLM-based Masking Impact on Leakage Metrics")
ax.legend()
ax.axhline(y=0.69, color='red', linestyle='--', alpha=0.5, label='Profit Mirage PC baseline')

# 标注 rule-based 和 LLM-based 的分界
n_rule = len(ablation_configs)
ax.axvline(x=n_rule - 0.5, color='blue', linestyle=':', alpha=0.5)
ax.text(n_rule - 0.5, ax.get_ylim()[1] * 0.95, '← rule-based | LLM-based →',
        ha='center', va='top', fontsize=9, color='blue')

plt.tight_layout()
plt.savefig(str(PROJECT_ROOT / 'data' / 'results' / 'ablation_metrics.png'), dpi=150, bbox_inches='tight')
plt.show()

print("\nMarginal contribution (delta-L per step, rule-based chain):")
rule_df = abl_df[abl_df["config"].isin(ablation_configs.keys())]
for i in range(1, len(rule_df)):
    delta = rule_df.iloc[i]["L"] - rule_df.iloc[i-1]["L"]
    print(f"  {rule_df.iloc[i-1]['config']} -> {rule_df.iloc[i]['config']}: dL = {delta:+.3f}")

## 2.5 实体掩码方式对比

三种实体处理方式的 PC 对比：
1. **占位符 `[实体N]`**（rule-based mask_entity）：直接替换为占位符
2. **LLM 自然模糊化**（llm mask_entity）：让 LLM 自然地替换实体信息
3. **虚构实体替换**（entity swap）：用完全虚构的实体名替换

PC 高 = 模型正确忽略了实体名称的变化（好），PC 低 = 模型依赖实体名（可能是记忆触发器）。

In [ ]:
# 生成虚构实体替换版本（entity swap）
print("生成虚构实体替换版本...")
swap_prompts = [entity_swap_prompt(tc.news.content) for tc in test_cases]
swap_responses = client.batch_chat_concurrent(swap_prompts)
swap_texts = [r.raw_response for r in swap_responses]

# 预览 entity swap 效果（随机抽取）
swap_samples = random.sample(list(zip(test_cases, swap_texts)), min(3, len(test_cases)))
for tc, swapped in swap_samples:
    show_comparison(tc.news.content, swapped, title=f"[{tc.id}] 虚构实体替换")

# 对三种方式分别打分
print("\n对三种实体处理方式分别打分...")

# 1. Rule-based entity mask
rule_entity_config = MaskingConfig(mask_entity=True)
rule_entity_texts = [apply_masking(tc.news.content, rule_entity_config, tc.key_entities, client=client) for tc in test_cases]
rule_entity_responses = run_scoring_batch(client, MaskingConfig(), rule_entity_texts, output_format)

# 2. LLM entity mask
llm_entity_config = MaskingConfig(mask_entity=True, mask_mode="llm")
llm_entity_texts = [apply_masking(tc.news.content, llm_entity_config, tc.key_entities, client=client) for tc in test_cases]
llm_entity_responses = run_scoring_batch(client, MaskingConfig(), llm_entity_texts, output_format)

# 3. Entity swap (already generated)
swap_entity_responses = run_scoring_batch(client, MaskingConfig(), swap_texts, output_format)

# 原始打分
orig_responses_for_entity = run_scoring_batch(client, MaskingConfig(), [tc.news.content for tc in test_cases], output_format)

# 计算 PC for each method
def calc_entity_pc(orig_resps, modified_resps):
    orig_dirs = [parse_5bin(r.raw_response)["direction"] for r in orig_resps]
    mod_dirs = [parse_5bin(r.raw_response)["direction"] for r in modified_resps]
    return prediction_consistency(orig_dirs, mod_dirs)

rule_pc = calc_entity_pc(orig_responses_for_entity, rule_entity_responses)
llm_pc = calc_entity_pc(orig_responses_for_entity, llm_entity_responses)
swap_pc = calc_entity_pc(orig_responses_for_entity, swap_entity_responses)

print(f"\n实体掩码方式对比 (PC):")
print(f"  占位符 [实体N] (rule-based): PC = {rule_pc:.3f}")
print(f"  LLM 自然模糊化:              PC = {llm_pc:.3f}")
print(f"  虚构实体替换 (entity swap):   PC = {swap_pc:.3f}")

fig, ax = plt.subplots(figsize=(8, 5))
methods = ["占位符\n[实体N]", "LLM\n自然模糊化", "虚构实体\n替换"]
pcs = [rule_pc, llm_pc, swap_pc]
colors = ["#2196F3", "#4CAF50", "#FF9800"]
bars = ax.bar(methods, pcs, color=colors)
ax.set_ylabel("PC (预测一致性)")
ax.set_title("实体掩码方式对比：PC 高 = 模型忽略实体名（好）")
ax.set_ylim(0, 1)
for bar, val in zip(bars, pcs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, f"{val:.3f}", ha='center')
plt.tight_layout()
plt.savefig(str(PROJECT_ROOT / 'data' / 'results' / 'entity_mask_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()

print("\n解读：PC 高意味着模型在实体名变化后仍给出相同预测，说明它基于新闻内容而非实体名做判断。")
print("PC 低则意味着实体名变化导致预测改变，说明实体名是记忆触发器。")

## 3. 安慰剂测试

用 LLM 生成自相矛盾的虚构新闻，使理性分析者无法得出确定方向。

**方法：** LLM 改写新闻为自相矛盾版本（如火锅企业宣布半导体产能扩张，同时加息和降息），保留句式结构但让内容无法分析。

**阈值推导：**
- 数据集分布：up=19, down=17, neutral=6
- 多数类猜测率 (majority class rate) = 19/42 = 45.2%
- 安慰剂阈值 = majority_class_rate + 10% = **55%**
- 通过标准：安慰剂准确率 < 55%（低于此阈值说明模型确实在分析而非记忆）

In [ ]:
# Generate placebo: LLM 改写为自相矛盾的虚构新闻
print("生成自相矛盾的安慰剂新闻...")
placebo_prompts = [placebo_rewrite_prompt(tc.news.content) for tc in test_cases]
placebo_responses_raw = client.batch_chat_concurrent(placebo_prompts)
placebo_texts = [r.raw_response for r in placebo_responses_raw]

# 随机抽取展示（重新运行刷新样本）
placebo_samples = random.sample(list(range(len(test_cases))), min(3, len(test_cases)))
for idx in placebo_samples:
    show_comparison(test_cases[idx].news.content, placebo_texts[idx],
                    title=f"[{test_cases[idx].id}] 原文 vs 安慰剂（自相矛盾版）")

# 并发调用 placebo + real baseline
config = MaskingConfig()
placebo_score_responses = run_scoring_batch(client, config, placebo_texts, output_format)
real_responses = run_scoring_batch(client, config, [tc.news.content for tc in test_cases], output_format)

placebo_results = []
for tc, resp in zip(test_cases, placebo_score_responses):
    parsed = parse_5bin(resp.raw_response)
    placebo_results.append({
        "case_id": tc.id, "expected": tc.expected_direction.value,
        "predicted": parsed["direction"],
        "correct": parsed["direction"] == tc.expected_direction.value,
    })

real_results = []
for tc, resp in zip(test_cases, real_responses):
    parsed = parse_5bin(resp.raw_response)
    real_results.append({
        "case_id": tc.id, "expected": tc.expected_direction.value,
        "predicted": parsed["direction"],
        "correct": parsed["direction"] == tc.expected_direction.value,
    })

## 4. 安慰剂测试结果

In [ ]:
real_acc = sum(r["correct"] for r in real_results) / len(real_results)
placebo_acc = sum(r["correct"] for r in placebo_results) / len(placebo_results)

# 阈值推导
n_up = sum(1 for tc in test_cases if tc.expected_direction.value == "up")
n_down = sum(1 for tc in test_cases if tc.expected_direction.value == "down")
n_neutral = sum(1 for tc in test_cases if tc.expected_direction.value == "neutral")
majority_rate = max(n_up, n_down, n_neutral) / len(test_cases)
threshold = majority_rate + 0.10

print(f"数据集分布: up={n_up}, down={n_down}, neutral={n_neutral}")
print(f"多数类猜测率: {majority_rate:.1%}")
print(f"安慰剂阈值 = {majority_rate:.1%} + 10% = {threshold:.1%}")
print()
print(f"Real data accuracy:    {real_acc:.1%}")
print(f"Placebo accuracy:      {placebo_acc:.1%}")
print(f"Difference:            {real_acc - placebo_acc:+.1%}")
print()

if placebo_acc < threshold:
    print(f"PASS: Placebo accuracy ({placebo_acc:.1%}) < threshold ({threshold:.1%})")
    print("  模型在自相矛盾新闻上准确率低，说明它确实在分析文本内容。")
else:
    print(f"FAIL: Placebo accuracy ({placebo_acc:.1%}) >= threshold ({threshold:.1%})")
    print("  模型在安慰剂数据上仍保持高准确率，存在记忆泄露风险。")

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(["Real", "Placebo"], [real_acc, placebo_acc], color=["#2196F3", "#9E9E9E"])
ax.axhline(y=threshold, color='red', linestyle='--', label=f'Threshold ({threshold:.0%})')
ax.axhline(y=0.33, color='green', linestyle='--', alpha=0.5, label='Random (33%)')
ax.axhline(y=majority_rate, color='orange', linestyle=':', alpha=0.5, label=f'Majority class ({majority_rate:.0%})')
ax.set_ylabel("Accuracy")
ax.set_title("Placebo Test: Real vs Self-contradictory Data")
ax.set_ylim(0, 1)
ax.legend()
for bar, val in zip(bars, [real_acc, placebo_acc]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, f"{val:.1%}", ha='center')
plt.tight_layout()
plt.savefig(str(PROJECT_ROOT / 'data' / 'results' / 'placebo_test.png'), dpi=150, bbox_inches='tight')
plt.show()

## 5. 保存结果

In [ ]:
output = {
    "ablation_metrics": abl_metrics,
    "entity_comparison": {
        "rule_based_pc": rule_pc,
        "llm_mask_pc": llm_pc,
        "entity_swap_pc": swap_pc,
    },
    "placebo": {
        "real_accuracy": real_acc,
        "placebo_accuracy": placebo_acc,
        "threshold": threshold,
        "majority_class_rate": majority_rate,
        "pass": placebo_acc < threshold,
    },
}
output_path = PROJECT_ROOT / 'data' / 'results' / 'ablation_results.json'
with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(output, f, ensure_ascii=False, indent=2, default=str)
print(f"Results saved to {output_path}")